# Soumission finale — recette unique (AIMS Africa Multilingual Tokenizer Challenge)

Notebook **linéaire et autonome** : entraîne **une** recette (la gagnante du balayage du
notebook 02), l'évalue avec la **métrique officielle**, la vérifie avec le **checker
officiel**, construit le dossier `submissions/<slug>/` complet et publie les artefacts.

- Recette actuelle : **`c12-alph500-bf`** — score réel **1.8359** (run Colab du 2026-09-10)
- Pour changer de recette : modifiez **uniquement** le dict `RECIPE` (§4) d'après le rapport
  `reports/optimization_sweep.{json,md}` produit par le notebook 02, puis **Tout exécuter**.
- Entraînement sur `train` uniquement ; évaluation sur `validation` ; aucune donnée externe.

## 1. Installation (versions officielles)

`tokenizers==0.22.1` est **imposé** par le challenge.

In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy sentencepiece
import tokenizers
print("tokenizers:", tokenizers.__version__, "(attendu 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Installer tokenizers==0.22.1 (exigence officielle)"

## 2. Métrique officielle (copie exacte du code du challenge)

In [ ]:
# =============================================================================
# 2. Constantes + métrique OFFICIELLES (compétition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Constantes officielles (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Dataset officiel ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Paramètres du balayage (modifiables) ----------------------------------
VOCAB_SIZE = 10_000            # imposé par le challenge
MAX_TRAIN_DOCS = None          # None = tout le train (240 000) ; ex. 60_000 pour un pré-balayage rapide
BASELINE_REFERENCE_SCORE = 2.059977   # score officiel obtenu par 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- MÉTRIQUE OFFICIELLE ---------------------------------------------------
def count_words(text: str) -> int:
    """Mots = séparés par des espaces blancs (comme l'évaluateur officiel)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id émis pour un texte non représentable (comme l'évaluateur officiel)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = liste de (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Score officiel = moyenne des langues notées (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"langues notées manquantes : {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x moyenne(fertility brute des langues notées)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


def validate_tokenizer_file(path):
    """Contrôles officiels de validation (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("fichier > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulaire {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("une langue ne produit aucun token")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("un décodage est vide")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Métrique officielle chargée. Vocab max:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

## 3. Données officielles (entraînement sur `train` uniquement)

In [ ]:
# =============================================================================
# 3. Chargement du dataset officiel + préparation des textes
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nEntraînement :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation   :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "train inattendu"
assert len(val_rows) == 24_000, "validation inattendue"

## 4. La recette — LE seul endroit à modifier

In [ ]:
# =============================================================================
# 4. LA RECETTE — LE seul endroit à modifier
#    Après chaque run du notebook 02 (balayage), recopiez ici la configuration
#    gagnante du rapport reports/optimization_sweep.{json,md}, puis relancez
#    tout le notebook (Runtime > Tout exécuter).
# =============================================================================
RECIPE = dict(
    name="c12-alph500-bf",                 # nom EXACT de la config gagnante (rapport réel)
    model="bpe",                           # bpe | wordpiece | unigram | sp_unigram | sp_bpe
    pre="whitespace_split",                # whitespace | whitespace_split | whitespace_split_punct | byte_level | metaspace
    min_freq=5,
    alphabet=False,                        # True = alphabet complet du train
    limit_alphabet=500,                    # None = illimité
    byte_fallback=True,                    # 256 tokens <0xXX> -> zéro [UNK]
    boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},   # sur-échantillonnage des langues notées
    # character_coverage=0.9995,            # (SentencePiece uniquement)
)
assert re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", RECIPE["name"]), "nom de config invalide"
print("Recette :", RECIPE["name"])

## 5. Fonctions d'entraînement (identiques au balayage du notebook 02)

In [ ]:
# =============================================================================
# 4. Définition des configurations candidates
# =============================================================================
from tokenizers.models import BPE, Unigram, WordPiece
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPre, Metaspace
from tokenizers import Regex
from tokenizers.pre_tokenizers import Sequence as PreSequence, Split
from tokenizers.pre_tokenizers import Whitespace, WhitespaceSplit
from tokenizers.decoders import ByteFallback, Sequence
from tokenizers.decoders import Metaspace as MetaspaceDec
from tokenizers.decoders import ByteLevel as ByteLevelDec
from tokenizers.trainers import BpeTrainer, UnigramTrainer, WordPieceTrainer

# --- EXP-004 : byte fallback ---------------------------------------------------
# Recette mesurée : les 256 tokens <0xXX> doivent être DANS le vocabulaire du modèle (sinon le
# byte_fallback du modèle ne trouve rien et émet [UNK]) et comptent DANS vocab_size.
# Passer par add_tokens() après entraînement NE fonctionne PAS (mesuré : 41/41 mots -> [UNK]).
BYTE_TOKENS = [f"<0x{i:02X}>" for i in range(256)]


def is_byte_token(token):
    """Vrai pour les tokens byte du type <0xEF> (tokens ordinaires, comme dans Llama-2)."""
    return len(token) == 6 and token.startswith("<0x") and token.endswith(">")


# =============================================================================
# 5. Balayage : entraînement + évaluation officielle
# =============================================================================
def corpus_iterator(train_by_lang, boost=None, log_every=50_000):
    """Itère les textes multilingues en round-robin (équilibré) avec sur-échantillonnage."""
    boost = boost or {}
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if log_every and i % log_every == 0:
                    print(f"    ... {i:,} textes fournis")


def strip_byte_added_tokens(tokenizer):
    """Rend les tokens byte ordinaires (forme des tokenizers Llama-2).

    Le trainer les ajoute via special_tokens (seul moyen mesuré de les faire entrer dans le
    vocabulaire du modèle) : on retire leur entrée `added_tokens`, ils restent des tokens du
    modèle BPE — byte fallback intact, et decode(skip_special_tokens=True) ne les jette plus.
    """
    payload = json.loads(tokenizer.to_str())
    payload["added_tokens"] = [t for t in payload["added_tokens"]
                               if not is_byte_token(t["content"])]
    return Tokenizer.from_str(json.dumps(payload))


SP_METASPACE = "\u2581"      # marqueur d'espace de SentencePiece (pas U+2581 litteral dans le code)


def build_sentencepiece_tokenizer(cfg):
    """Entraîne SentencePiece puis le convertit en tokenizer.json HuggingFace.

    Conversion mesurée localement : les pièces SentencePiece (dont les 256 <0xXX> du
    byte_fallback) deviennent un modèle Unigram, avec un pré-tokeniseur/décodeur Metaspace
    — le checker officiel valide le fichier obtenu (valid=True, vocab 10 000).
    """
    import sentencepiece as spm

    model_type = "unigram" if cfg["model"] == "sp_unigram" else "bpe"
    sp_dir = OUTPUT_ROOT / "sentencepiece"
    sp_dir.mkdir(parents=True, exist_ok=True)
    prefix = sp_dir / cfg["name"]
    spm.SentencePieceTrainer.train(
        sentence_iterator=corpus_iterator(train_by_lang, cfg.get("boost")),
        model_prefix=str(prefix), vocab_size=VOCAB_SIZE, model_type=model_type,
        byte_fallback=True,
        character_coverage=cfg.get("character_coverage", 0.9995),
        normalization_rule_name="identity",      # aucun NFKC/NFKD, aucune conversion ASCII
        # hard_vocab_limit=False : mesuré localement, sinon SentencePiece peut boucler sans fin
        # quand l'alphabet est grand et le corpus insuffisant pour remplir 10 000 pièces.
        hard_vocab_limit=False,
        num_threads=4, minloglevel=2)
    sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    vocab = [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())]
    tokenizer = Tokenizer(Unigram(vocab=vocab, unk_id=sp.unk_id(), byte_fallback=True))
    tokenizer.pre_tokenizer = Metaspace(replacement=SP_METASPACE, prepend_scheme="always",
                                        split=True)
    tokenizer.decoder = Sequence([
        ByteFallback(),
        MetaspaceDec(replacement=SP_METASPACE, prepend_scheme="always"),
    ])
    print(f"    SentencePiece {model_type} : {sp.get_piece_size():,} pièces "
          f"(dont byte fallback), converti en Unigram + Metaspace")
    return tokenizer


def build_tokenizer(cfg):
    """Construit et entraîne un tokenizer selon la configuration (utilise train seulement)."""
    if cfg["model"] in ("sp_unigram", "sp_bpe"):
        return build_sentencepiece_tokenizer(cfg)

    byte_fallback = bool(cfg.get("byte_fallback", False))
    if cfg["model"] == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=byte_fallback))
    elif cfg["model"] == "wordpiece":
        tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    else:
        tokenizer = Tokenizer(Unigram())
    tokenizer.normalizer = NFC()

    if cfg["pre"] == "whitespace":
        tokenizer.pre_tokenizer = Whitespace()
    elif cfg["pre"] == "whitespace_split":
        tokenizer.pre_tokenizer = WhitespaceSplit()
    elif cfg["pre"] == "whitespace_split_punct":
        # EXP-008 : mots propres — la ponctuation devient un token à part (behavior=isolated),
        # les merges apprennent les vraies fréquences de mots au lieu de formes « mot, ».
        tokenizer.pre_tokenizer = PreSequence([WhitespaceSplit(),
                                               Split(Regex(r"\p{P}"), behavior="isolated")])
    elif cfg["pre"] == "byte_level":
        tokenizer.pre_tokenizer = ByteLevelPre(add_prefix_space=False, use_regex=True)
        tokenizer.decoder = ByteLevelDec()

    if cfg["alphabet"] == "bytes":
        alphabet = ByteLevelPre.alphabet()
    elif cfg["alphabet"] is True:
        alphabet = train_alphabet
    else:
        alphabet = None

    kwargs = dict(vocab_size=VOCAB_SIZE, min_frequency=cfg["min_freq"],
                  special_tokens=["[UNK]"] + (BYTE_TOKENS if byte_fallback else []))
    if alphabet:
        kwargs["initial_alphabet"] = alphabet
    # limit_alphabet : ne garder que les N caractères les plus fréquents du train ; les
    # caractères écartés sont couverts par le byte fallback (aucun [UNK]).
    if cfg.get("limit_alphabet"):
        kwargs["limit_alphabet"] = cfg["limit_alphabet"]
    if cfg["model"] == "bpe":
        trainer = BpeTrainer(**kwargs)
    elif cfg["model"] == "wordpiece":
        trainer = WordPieceTrainer(**kwargs)
    else:
        kwargs.pop("min_frequency", None)      # UnigramTrainer n'accepte pas min_frequency
        kwargs["unk_token"] = "[UNK]"
        trainer = UnigramTrainer(**kwargs)

    tokenizer.train_from_iterator(
        corpus_iterator(train_by_lang, cfg.get("boost")), trainer=trainer)

    if byte_fallback:
        tokenizer.decoder = ByteFallback()
        tokenizer = strip_byte_added_tokens(tokenizer)
        print(f"    byte_fallback : {len(BYTE_TOKENS)} tokens <0xXX> dans le vocabulaire "
              f"(total {tokenizer.get_vocab_size(with_added_tokens=True):,}), "
              f"added_tokens restants = {len(json.loads(tokenizer.to_str())['added_tokens'])}")
    return tokenizer


## 6. Entraînement + évaluation officielle

In [ ]:
# =============================================================================
# 6. Entraînement de LA recette + évaluation officielle (métrique du challenge)
# =============================================================================
t0 = time.time()
tokenizer = build_tokenizer(RECIPE)
train_seconds = time.time() - t0

fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tokenizer)
score = competition_score(fertility, unk_rate)
raw, budget, breaches = guardrail(fertility)
penalised = penalised_scores(fertility, unk_rate)
vocab_size = tokenizer.get_vocab_size(with_added_tokens=True)

# même forme que les lignes du balayage -> les cellules suivantes sont identiques au notebook 02
best = dict(name=RECIPE["name"], note="recette de soumission (notebook 03)", config=RECIPE,
            vocab_size=vocab_size, train_seconds=train_seconds, score=score,
            fertility=fertility, unk_rate=unk_rate, penalised=penalised,
            tokens=tokens, words=words, unk_total=unk_total, lossy_rows=lossy,
            raw_scored=raw, guardrail_budget=budget, guardrail_breaches=breaches,
            guardrail_pass=not breaches, tokenizer=tokenizer)
results = [best]

print(f"RECETTE  {best['name']}")
print(f"  score     : {score:.4f}  (baseline {BASELINE_REFERENCE_SCORE:.4f}, "
      f"gain {BASELINE_REFERENCE_SCORE - score:+.4f})")
print(f"  vocab     : {vocab_size:,} | UNK : {unk_total} | lignes lossy : {lossy:,}")
print(f"  fertility : " + ", ".join(f"{l} {fertility[l]:.4f}" for l in LANGUAGES))
print(f"  guardrail : {'PASS' if not breaches else 'FAIL ' + str(breaches)}"
      f"  (budget {budget:.4f} | en {fertility['en']:.4f} | fr {fertility['fr']:.4f})")
assert vocab_size <= MAX_VOCAB_SIZE, "vocabulaire > 10 000"
assert not breaches, "guardrail EN/FR dépassé : ne pas soumettre cette recette telle quelle"
assert unk_total == 0, "[UNK] présents : vérifiez byte_fallback / alphabet (pénalité 100/UNK)"

## 7. Sauvegarde du modèle + rapport

In [ ]:
# =============================================================================
# 7. Sauvegarde du modèle + rapport (reports/submission_final.{json,md})
# =============================================================================
from datetime import datetime, UTC

best_model_dir = MODEL_DIR / f"optimized_{best['name']}"
best_model_dir.mkdir(parents=True, exist_ok=True)
best_tokenizer_path = best_model_dir / "tokenizer.json"
best["tokenizer"].save(str(best_tokenizer_path))

candidate_path = OUTPUT_ROOT / "tokenizer.json"          # candidat pour le checker
shutil.copy2(best_tokenizer_path, candidate_path)

row = {k: v for k, v in best.items() if k != "tokenizer"}
report = {
    "experiment": "submission_final",
    "date": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_rows": sum(len(v) for v in train_by_lang.values()),
                "validation_rows": len(val_rows),
                "word_definition": "len(text.split())"},
    "baseline_reference_score": BASELINE_REFERENCE_SCORE,
    "best": {k: row[k] for k in ("name", "score", "config", "guardrail_pass")},
    "results": [row],
}
(REPORT_DIR / "submission_final.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

md = ["# Soumission finale — métrique officielle", "",
      f"*Dataset `{DATASET_NAME}` @ `{DATASET_REVISION}` — train : "
      f"{sum(len(v) for v in train_by_lang.values()):,} textes, validation : {len(val_rows):,} lignes.*", "",
      "## Résultat", "",
      f"- **Score (moyenne ha/sw/yo/am) : {best['score']:.4f}** — baseline {BASELINE_REFERENCE_SCORE:.4f}, "
      f"gain {BASELINE_REFERENCE_SCORE - best['score']:+.4f}",
      f"- Guardrail EN/FR : {'PASS' if best['guardrail_pass'] else 'FAIL'} "
      f"(budget {best['guardrail_budget']:.4f}, en {fertility['en']:.4f}, fr {fertility['fr']:.4f})",
      f"- UNK émis sur la validation : {best['unk_total']}",
      f"- Modèle : `{best_tokenizer_path.name}` ({best_tokenizer_path.stat().st_size:,} octets)", "",
      "## Détail par langue", "",
      "| Langue | Fertility | UNK rate | Pénalisé |", "|---|---:|---:|---:|"]
for l in LANGUAGES:
    md.append(f"| {l} | {fertility[l]:.4f} | {unk_rate.get(l, 0.0):.6f} | {penalised[l]:.4f} |")
md += ["", "## Configuration", "", "```json", json.dumps(RECIPE, indent=2, ensure_ascii=False), "```", ""]
(REPORT_DIR / "submission_final.md").write_text("\n".join(md), encoding="utf-8")
print("Modèle :", best_tokenizer_path, f"({best_tokenizer_path.stat().st_size:,} octets)")
print("Rapports :", REPORT_DIR / "submission_final.json", "|", REPORT_DIR / "submission_final.md")

## 8. Vérification avec le **checker officiel** du challenge

In [ ]:
# =============================================================================
# 9. Checker officiel (starter/utils.py du dépôt du challenge)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("utils.py officiel téléchargé")
    except Exception as exc:
        print("Téléchargement impossible :", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Checker officiel indisponible — utilisation des contrôles intégrés.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Contrôles intégrés complémentaires (équivalents à competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nContrôles officiels intégrés :", checks["checks"])
print("Erreurs :", checks["errors"] or "aucune")
print("Langues non lossless (round-trip) :", checks["lossy_languages"] or "aucune (lossless)")

## 9. Dossier de soumission `submissions/<slug>/`

In [ ]:
# =============================================================================
# 10. Génération du dossier de soumission
# =============================================================================
import re
import shutil

# --- Métadonnées (participation individuelle) -------------------------------
TEAM_NAME = "Maick Dane Nkou"           # <= 80 caractères, apparaît sur le leaderboard
MEMBERS = ["Maick Dane Nkou"]           # <= 6 membres
AFFILIATION = "AIMS SOUTH AFRICA"       # optionnel (<= 120 caractères)
SLUG = "maick-dane-nkou"                # dossier : minuscules kebab-case (aligné sur TEAM_NAME)
NOTEBOOK_GLOBS = [                      # où chercher notebook.ipynb, dans l'ordre
    "*.ipynb",                                        # Colab : à côté du notebook en cours
    "notebooks/03_submission_final.ipynb",          # dépôt cloné (ce notebook)
]
NOTEBOOK_URL = (                        # dernier recours : téléchargement depuis le dépôt
    "https://raw.githubusercontent.com/maick-code/tokenizer/"
    "arena/01a0889d-tokenizer/notebooks/03_submission_final.ipynb"
)

assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", SLUG), "slug invalide (minuscules kebab-case)"
assert SLUG != "baseline", "le dossier 'baseline' est réservé"
assert len(MEMBERS) <= 6, "6 membres maximum"
assert TEAM_NAME.strip() and len(TEAM_NAME.strip()) <= 80, "nom d'équipe invalide"

submission_dir = SUBMISSIONS_DIR / SLUG

# Nettoyage : le checker officiel refuse tout fichier inattendu
ALLOWED_FILES = {"tokenizer.json", "metadata.yml", "notebook.ipynb", "README.md"}
if submission_dir.is_dir():
    for item in sorted(submission_dir.iterdir()):
        if item.name not in ALLOWED_FILES:
            print("supprimé (fichier non autorisé) :", item)
            shutil.rmtree(item) if item.is_dir() else item.unlink()
submission_dir.mkdir(parents=True, exist_ok=True)

# --- tokenizer.json ---------------------------------------------------------
shutil.copy2(best_tokenizer_path, submission_dir / "tokenizer.json")

# --- description exacte de la configuration gagnante ------------------------
PRE_NAMES = {"whitespace_split": "WhitespaceSplit", "whitespace": "Whitespace",
             "byte_level": "ByteLevel", "metaspace": "SentencePiece Metaspace",
             "whitespace_split_punct": "WhitespaceSplit + punctuation isolated"}
MODEL_NAMES = {"bpe": "BPE", "unigram": "Unigram", "wordpiece": "WordPiece",
               "sp_unigram": "SentencePiece unigram", "sp_bpe": "SentencePiece BPE"}
pre_name = PRE_NAMES.get(best["config"].get("pre"), best["config"].get("pre"))
model_name = MODEL_NAMES.get(best["config"].get("model"), best["config"].get("model"))
is_sp = best["config"].get("model", "").startswith("sp_")
normalizer_name = ("none (SentencePiece identity normalization)" if is_sp else "NFC")
boost = best["config"].get("boost") or {}
_boost_levels = sorted(set(boost.values()), reverse=True)
if len(_boost_levels) <= 1:
    oversample = (f" with {'/'.join(sorted(boost))} oversampled "
                  f"x{_boost_levels[0]}" if boost else "")
else:
    _parts = " + ".join(f"{'/'.join(sorted(l for l, b in boost.items() if b == m))} x{m}"
                        for m in _boost_levels)
    oversample = f" with {_parts} oversampled"
byte_clause = (", byte fallback (256 <0xXX> tokens, no [UNK])"
               if best["config"].get("byte_fallback") else "")
alphabet_clause = (f", alphabet limited to the {best['config']['limit_alphabet']:,} most frequent "
                   f"characters" if best["config"].get("limit_alphabet") else "")
# Le checker plafonne `approach` à 240 caractères : on réserve la fin (score + guardrail), puis
# on ajoute les éléments descriptifs tant qu'il reste de la place — jamais de coupe en pleine
# phrase, et jamais de parenthèse laissée ouverte.
approach_head = f"{model_name} {round(best['vocab_size'] / 1000)}k ({best['name']})"
approach_tail = (f"validation {best['score']:.4f} (baseline {BASELINE_REFERENCE_SCORE:.4f}), "
                 f"EN/FR guardrail {'PASS' if best['guardrail_pass'] else 'FAIL'}")
approach = approach_head
for extra in (f"{normalizer_name}, {pre_name} pre-tokenization{byte_clause}{alphabet_clause}",
              f"official train split only{oversample}"):
    if len(approach) + 2 + len(extra) + 2 + len(approach_tail) <= 240:
        approach = f"{approach}, {extra}"
approach = f"{approach}, {approach_tail}"
assert len(approach) <= 240, f"approach : {len(approach)} caracteres (> 240)"
assert approach.endswith(("PASS", "FAIL")), "approach tronque"

# --- metadata.yml -----------------------------------------------------------
metadata = {
    "team": TEAM_NAME,
    "members": MEMBERS,
    "affiliation": AFFILIATION[:120],
    "approach": approach,
}
try:
    import yaml
    (submission_dir / "metadata.yml").write_text(
        yaml.safe_dump(metadata, allow_unicode=True, sort_keys=False), encoding="utf-8")
    print("metadata.yml écrit (YAML).")
except ImportError:
    lines = [f"team: {TEAM_NAME}", "members:"]
    lines += [f"  - {member}" for member in MEMBERS]
    lines += [f"affiliation: {AFFILIATION}", f"approach: {approach}"]
    (submission_dir / "metadata.yml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("metadata.yml écrit (YAML manuel).")

# --- README.md (approche) ---------------------------------------------------
unk_name = "<unk>" if is_sp else "[UNK]"
if best["config"].get("pre") in ("whitespace_split", "metaspace", "byte_level"):
    pretok_sentence = ("\u2014 punctuation stays attached to its word, so no token is spent "
                       "on isolated `,` `.` `)` \u2026\n")
elif best["config"].get("pre") == "whitespace_split_punct":
    pretok_sentence = ("\u2014 punctuation is isolated into its own tokens (`\\p{P}`), so merges "
                       "learn clean word frequencies\n")
elif best["config"].get("pre") == "whitespace":
    pretok_sentence = "\u2014 splits on whitespace only\n"
else:
    pretok_sentence = "\n"
pretok_line = f"- Pre-tokenizer: `{pre_name}` {pretok_sentence}"
alphabet_line = (
    f"- Alphabet: limited to the {best['config']['limit_alphabet']:,} most frequent train "
    f"characters (`limit_alphabet`); the characters left out are still covered by the byte "
    f"fallback, so no `[UNK]` appears\n"
    if best["config"].get("limit_alphabet") else "")
size_of_byte_vocab = ("yes — the 256 `<0xXX>` tokens cover every possible UTF-8 character, so\n"
                     "  no `[UNK]` can ever be emitted" if best["config"].get("byte_fallback")
                     else "no — the few `[UNK]` left are rare non-African residues of the source\n"
                          "  text (Arabic presentation forms, CJK, kana, hangul, emoji)")
result_rows = "\n".join(
    f"| {LANGUAGE_NAMES[l]} | {best['fertility'][l]:.4f} | {best['unk_rate'][l]:.6f} | "
    f"{best['penalised'][l]:.4f} |" for l in LANGUAGES)
(submission_dir / "README.md").write_text(
    f"# {TEAM_NAME}\n\n"
    f"Individual entry — tokenizer `{best['name']}`.\n\n"
    f"## Approach\n\n"
    f"- Model: {model_name} with `{unk_name}` as unknown token; vocabulary "
    f"{best['vocab_size']:,} / 10,000\n"
    f"- Normalizer: {normalizer_name}\n"
    f"{alphabet_line}"
    f"{pretok_line}"
    f"- Byte fallback: {size_of_byte_vocab}\n"
    f"- Training corpus: official `train` split only, balanced round-robin over the six\n"
    f"  languages{oversample}; no external corpus and no pre-trained tokenizer\n"
    f"- Post-processor: none | Decoder: "
    f"{'ByteFallback' if best['config'].get('byte_fallback') else 'none'}\n"
    f"- Built with `tokenizers==0.22.1`\n\n"
    f"## Results (official validation split, official metric)\n\n"
    f"| Language | Fertility | UNK rate | Score |\n|---|---:|---:|---:|\n{result_rows}\n\n"
    f"- **Score (mean of ha, sw, yo, am): {best['score']:.4f}** — baseline BPE 10k "
    f"{BASELINE_REFERENCE_SCORE:.4f}, i.e. a gain of "
    f"{BASELINE_REFERENCE_SCORE - best['score']:+.4f} "
    f"({100 * (BASELINE_REFERENCE_SCORE - best['score']) / BASELINE_REFERENCE_SCORE:.1f} %)\n"
    f"- Context guardrail EN/FR: **{'PASS' if best['guardrail_pass'] else 'FAIL'}** "
    f"(budget {best['guardrail_budget']:.4f}, en {best['fertility']['en']:.4f}, "
    f"fr {best['fertility']['fr']:.4f})\n"
    f"- UNK emitted on validation: {best['unk_total']}\n\n"
    f"## Files\n\n"
    f"- `tokenizer.json` — the submitted tokenizer\n"
    f"- `metadata.yml` — team metadata\n"
    f"- `notebook.ipynb` — the notebook that built this tokenizer "
    f"(`notebooks/02_optimization_sweep.ipynb`)\n"
    f"- `README.md` — this file\n",
    encoding="utf-8")

# --- notebook.ipynb (obligatoire avant la date limite) ----------------------
def attach_notebook(target_dir):
    """Copie le notebook depuis le disque (Colab puis dépôt cloné), sinon le télécharge."""
    for pattern in NOTEBOOK_GLOBS:
        matches = sorted(OUTPUT_ROOT.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            shutil.copy2(matches[0], target_dir / "notebook.ipynb")
            print(f"notebook.ipynb : copié depuis {matches[0]}")
            return True
    try:
        import urllib.request
        urllib.request.urlretrieve(NOTEBOOK_URL, target_dir / "notebook.ipynb")
        print("notebook.ipynb : téléchargé depuis le dépôt")
        return True
    except Exception as exc:
        print("notebook.ipynb MANQUANT :", exc)
        print("  -> un dossier sans notebook.ipynb est DISQUALIFIÉ (règlement du challenge)")
        print("  -> déposez le notebook dans", target_dir)
        return False


notebook_attached = attach_notebook(submission_dir)

print("\nDossier de soumission :", submission_dir)
for path in sorted(submission_dir.iterdir()):
    print(f"  {path.name} ({path.stat().st_size:,} octets)")
print("\nFichiers autorisés :", sorted(ALLOWED_FILES))
print("metadata.yml :", metadata)
print("Prêt pour la PR officielle :", "OUI" if notebook_attached else "presque (notebook.ipynb à ajouter)")


## 10. Récapitulatif + checklist de la PR officielle

In [ ]:
# =============================================================================
# 10. Récapitulatif + checklist de la PR officielle
# =============================================================================
print("Soumission prête.\\n")
print(f"  recette        : {best['name']} -> score {best['score']:.4f} "
      f"(baseline {BASELINE_REFERENCE_SCORE:.4f})")
print(f"  guardrail      : {'PASS' if best['guardrail_pass'] else 'FAIL'}")
print(f"  dossier        : {submission_dir}")
if official_report is not None:
    print(f"  checker        : {'READY FOR SUBMISSION' if official_report.get('valid') else 'NOT READY -> ' + str(official_report.get('errors'))}")
print("""
CHECKLIST PR OFFICIELLE (dépôt AIMS) :
  1. fork de https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge
  2. branche du fork nommée EXACTEMENT `submission`
  3. copier le dossier submissions/maick-dane-nkou/ à la racine du fork (4 fichiers)
  4. ouvrir la PR vers le dépôt officiel -> la GitHub Action valide et classe
""")

## 11. Publier les artefacts sur GitHub (token masqué)

Même méthode que le notebook 02 : la cellule suivante **écrit** le script
`push_artifacts_to_github.py`, puis une cellule l'exécute **dans le processus** (le token
n'est **jamais affiché** : champ masqué `getpass`).

### Cellule « script de publication »

In [ ]:
%%writefile push_artifacts_to_github.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Publie les artefacts du challenge vers votre dépôt GitHub (méthode 2 : token).

Le token est fourni par saisie **masquée** (recommandé), par variable
d'environnement, ou par le secret Colab ``GITHUB_TOKEN``. Il n'est **jamais**
affiché, jamais écrit sur disque, jamais commité : toutes les sorties passent par
``redact()``.

Ce qui est publié (par défaut) :
    models/**      tokenizer(s) entraîné(s)
    reports/**     rapports JSON / Markdown
    submissions/** (avec --include-submissions) dossier de soumission

Exemples
--------
Colab — recommandé (dans une cellule Python, champ masqué actif) :
    import sys, runpy
    sys.argv = ["push_artifacts_to_github.py", "--source", "/content", "--include-submissions"]
    try:
        runpy.run_path("/content/push_artifacts_to_github.py", run_name="__main__")
    except SystemExit as exc:
        print("code de sortie :", exc.code)

Colab — avec !python : un sous-processus n'a ni champ masqué ni Secrets, il faut
fournir le token autrement (secret exporté dans l'environnement, ou --token-file) :
    !python scripts/push_artifacts_to_github.py --source /content

Colab, en incluant le dossier de soumission :
    !python scripts/push_artifacts_to_github.py --source /content --include-submissions

Supprimer au passage un dossier de soumission obsolète (ancien slug) :
    python scripts/push_artifacts_to_github.py --source /content --include-submissions \\
        --prune-submissions

Local :
    python scripts/push_artifacts_to_github.py --repo . --source .

Vérifier sans rien publier :
    python scripts/push_artifacts_to_github.py --source . --no-push

Créer explicitement une branche inexistante :
    python scripts/push_artifacts_to_github.py --source . --branch nouvelle-branche --create-branch

Publier sur une autre branche / un autre dépôt :
    python scripts/push_artifacts_to_github.py --source . --branch main \
        --repo-url https://github.com/<user>/<repo>.git
"""

from __future__ import annotations

import argparse
import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/maick-code/tokenizer.git"
DEFAULT_BRANCH = "arena/01a0889d-tokenizer"   # branche de travail (main reste intacte)
DEFAULT_MESSAGE = "Artifacts: tokenizer.json + reports (run Colab)"
ARTIFACT_DIRS = ("models", "reports")
EXCLUDE_DIR_NAMES = {"__pycache__", ".ipynb_checkpoints", ".git"}
EXCLUDE_SUFFIXES = (".pyc", ".pyo", ".zip", ".tmp", ".log")

EXIT_OK, EXIT_ERROR, EXIT_MISSING, EXIT_UNSAFE = 0, 1, 2, 3


# --------------------------------------------------------------------------- #
# Utilitaires
# --------------------------------------------------------------------------- #
def log(message: str = "") -> None:
    print(message, flush=True)


def die(message: str, code: int) -> "NoReturn":  # noqa: F821
    log(f"\nERREUR : {message}")
    raise SystemExit(code)


def redact(text: str, token: str | None) -> str:
    """Supprime toute trace du token d'une sortie."""
    if not text:
        return ""
    if token:
        text = text.replace(token, "***")
    return text


def clone_dir_default() -> Path:
    if os.path.isdir("/content"):          # Google Colab
        return Path("/content/tokenizer")
    return Path.cwd() / ".push_clone"


# --------------------------------------------------------------------------- #
# Token
# --------------------------------------------------------------------------- #
def token_from_colab_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get("GITHUB_TOKEN")
        return value.strip() if value else None
    except Exception:
        return None


_COLAB_MASKED_FIELD_JS = r"""
new Promise((resolve) => {
  const box = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;margin-top:6px;'
                    + 'border:1px solid #c8c8c8;border-radius:6px;display:inline-block';
  const label = document.createElement('span');
  label.textContent = 'Colle ton token GitHub puis valide : ';
  const input = document.createElement('input');
  input.type = 'password';
  input.style.cssText = 'font-size:14px;padding:3px 5px;width:330px';
  const button = document.createElement('button');
  button.textContent = 'Enregistrer';
  button.style.cssText = 'margin-left:8px;padding:3px 12px';
  const done = () => {
    input.disabled = true; button.disabled = true;
    const value = input.value; box.remove(); resolve(value);
  };
  button.addEventListener('click', done);
  input.addEventListener('keydown', (event) => { if (event.key === 'Enter') done(); });
  box.appendChild(label); box.appendChild(input); box.appendChild(button);
  document.body.appendChild(box);
  input.focus();
})
"""


def token_from_colab_masked_field() -> str | None:
    """Champ de saisie masqué natif Colab (nécessite d'exécuter le script EN PROCESSUS).

    Fonctionne quand le script est lancé dans une cellule Python (``runpy``), pas
    avec ``!python`` : un sous-processus n'a pas accès à l'interface du notebook.
    """
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        value = output.eval_js(_COLAB_MASKED_FIELD_JS)
    except Exception as exc:
        log(f"Champ masqué Colab indisponible ({type(exc).__name__}) : repli sur la saisie classique.")
        return None
    if isinstance(value, str) and value.strip():
        log("Token saisi dans le champ masqué Colab (non affiché).")
        return value.strip().strip('"').strip("'")
    return None


def read_token(args: argparse.Namespace) -> str | None:
    """Token par ordre de priorité : --token-file, env, secret Colab, champ masqué Colab, saisie."""
    if args.token_file:
        path = Path(args.token_file)
        if not path.is_file():
            die(f"fichier de token introuvable : {path}", EXIT_ERROR)
        token = path.read_text(encoding="utf-8").strip()
        if token:
            log("Token lu depuis le fichier indiqué (--token-file).")
            return token

    for var in ("GITHUB_TOKEN", "GH_TOKEN"):
        token = os.environ.get(var)
        if token:
            log(f"Token récupéré depuis la variable d'environnement {var}.")
            return token.strip()

    token = token_from_colab_secret()
    if token:
        log("Token récupéré depuis le secret Colab 'GITHUB_TOKEN'.")
        return token

    if args.no_input:
        return None

    token = token_from_colab_masked_field()
    if token:
        return token

    prompt = "Colle ton token GitHub puis Entrée : "
    try:
        token = getpass.getpass(prompt)          # saisie masquée
    except Exception:
        try:
            token = input(prompt)                # repli si getpass indisponible
        except Exception:
            return None
    token = (token or "").strip().strip('"').strip("'")
    if token:
        log(f"Token saisi ({len(token)} caractères, non affiché).")
    return token or None


def authed_url(url: str, token: str | None) -> str:
    """URL https porteuse du token, uniquement pour github.com."""
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@")
    return url


# --------------------------------------------------------------------------- #
# Git
# --------------------------------------------------------------------------- #
def git(repo: Path | str | None, *args: str) -> subprocess.CompletedProcess:
    command = ["git"]
    if repo is not None:
        command += ["-C", str(repo)]
    return subprocess.run(command + list(args), capture_output=True, text=True)


def git_or_die(repo: Path | str | None, token: str | None, *args: str,
               what: str = "commande git") -> subprocess.CompletedProcess:
    result = git(repo, *args)
    if result.returncode != 0:
        die(f"{what} a échoué :\n{redact(result.stderr or result.stdout, token).strip()}",
            EXIT_ERROR)
    return result


# --------------------------------------------------------------------------- #
# Artefacts
# --------------------------------------------------------------------------- #
def best_model_tokenizer(source: Path) -> Path | None:
    """Chemin du tokenizer de la meilleure configuration du dernier balayage."""
    import json

    sweep = source / "reports" / "optimization_sweep.json"
    if not sweep.is_file():
        return None
    try:
        name = json.loads(sweep.read_text(encoding="utf-8"))["best"]["name"]
    except Exception:
        return None
    candidate = source / "models" / f"optimized_{name}" / "tokenizer.json"
    return candidate if candidate.is_file() else None


def prune_obsolete_submissions(source: Path) -> list[str]:
    """Supprime les dossiers submissions/<slug>/ obsolètes (tokenizer != meilleur modèle).

    Cas typique : après avoir renommé le SLUG, l'ancien dossier reste sur le disque et serait
    publié avec le nouveau — or le checker officiel exige exactement un répertoire de
    soumission. Seuls des dossiers dont le tokenizer.json diffère du meilleur modèle sont
    supprimés, et seulement s'il en reste plusieurs : le dossier courant est toujours conservé.
    """
    import hashlib

    subs = source / "submissions"
    if not subs.is_dir():
        return []
    dirs = sorted(p for p in subs.iterdir() if p.is_dir() and (p / "tokenizer.json").is_file())
    if len(dirs) < 2:
        return []
    best = best_model_tokenizer(source)
    digest = (lambda p: hashlib.sha256(p.read_bytes()).hexdigest())
    if best is not None and any(digest(d / "tokenizer.json") == digest(best) for d in dirs):
        keep = [d for d in dirs if digest(d / "tokenizer.json") == digest(best)]
    else:
        keep = [max(dirs, key=lambda d: d.stat().st_mtime)]
    removed = []
    for d in dirs:
        if d not in keep:
            shutil.rmtree(d)
            removed.append(d.name)
    if removed:
        log(f"dossiers de soumission obsolètes supprimés : {', '.join(removed)}")
        log(f"dossier conservé : {keep[0].name}")
    return removed


def collect_artifacts(source: Path, include_submissions: bool) -> list[str]:
    """Chemins relatifs (posix) des fichiers à publier, triés."""
    roots = list(ARTIFACT_DIRS) + (["submissions"] if include_submissions else [])
    files: list[str] = []
    for root in roots:
        base = source / root
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file():
                continue
            parts = set(path.relative_to(source).parts)
            if parts & EXCLUDE_DIR_NAMES or path.name.startswith("."):
                continue
            if path.suffix.lower() in EXCLUDE_SUFFIXES:
                continue
            files.append(path.relative_to(source).as_posix())
    return files


def safety_checks(source: Path, files: list[str], force: bool) -> list[str]:
    """Contrôles avant publication. Retourne la liste des avertissements bloquants."""
    import json

    problems: list[str] = []

    baseline = source / "reports" / "baseline_bpe_10k.json"
    if baseline.is_file():
        try:
            status = json.loads(baseline.read_text(encoding="utf-8")).get("status")
            if status != "computed_on_official_dataset":
                problems.append(
                    f"reports/baseline_bpe_10k.json : status = {status!r} "
                    "(run non conforme au dataset officiel)")
        except Exception as exc:
            problems.append(f"reports/baseline_bpe_10k.json illisible : {exc}")

    sweep = source / "reports" / "optimization_sweep.json"
    if sweep.is_file():
        try:
            payload = json.loads(sweep.read_text(encoding="utf-8"))
            rows = (payload.get("dataset") or {}).get("validation_rows")
            if rows != 24_000:
                problems.append(
                    f"reports/optimization_sweep.json : validation_rows = {rows} "
                    "(attendu 24 000 : le balayage n'a pas tourné sur le vrai dataset)")
        except Exception as exc:
            problems.append(f"reports/optimization_sweep.json illisible : {exc}")

    if not any(f.startswith("models/") and f.endswith("tokenizer.json") for f in files):
        problems.append("aucun models/**/tokenizer.json trouvé dans les artefacts")

    # Une soumission = UN dossier. Un dossier obsolète laissé par une exécution antérieure
    # (ancien slug, par exemple après avoir renommé SLUG) rendrait la PR invalide : le
    # checker officiel exige exactement un répertoire `submissions/<slug>/`.
    subs = source / "submissions"
    if subs.is_dir():
        slugs = sorted(p.name for p in subs.iterdir()
                       if p.is_dir() and (p / "tokenizer.json").is_file())
        if len(slugs) > 1:
            problems.append(
                f"plusieurs dossiers de soumission dans submissions/ : {', '.join(slugs)} "
                "(un seul slug est autorisé par PR ; supprimez les dossiers obsolètes)")

    if problems and not force:
        log("\n" + "!" * 74)
        log("PUBLICATION REFUSÉE — les artefacts semblent ne pas venir d'un run réel :")
        for problem in problems:
            log(f"  - {problem}")
        log("Corrigez le run, ou relancez avec --force pour publier quand même.")
        log("!" * 74)
        raise SystemExit(EXIT_UNSAFE)

    if problems:
        log("\nAVERTISSEMENT (--force) :")
        for problem in problems:
            log(f"  - {problem}")
    return problems


# --------------------------------------------------------------------------- #
# Programme principal
# --------------------------------------------------------------------------- #
def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Publie models/ et reports/ vers votre dépôt GitHub (méthode token).",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--source", default="/content" if os.path.isdir("/content") else ".",
                        help="répertoire contenant models/ et reports/ (défaut : /content ou .)")
    parser.add_argument("--repo", default=None,
                        help="clone git existant du dépôt (sinon clonage automatique)")
    parser.add_argument("--repo-url", default=DEFAULT_REPO_URL, help="URL https du dépôt")
    parser.add_argument("--branch", default=DEFAULT_BRANCH, help="branche cible")
    parser.add_argument("--message", default=DEFAULT_MESSAGE, help="message de commit")
    parser.add_argument("--token-file", default=None,
                        help="lire le token depuis un fichier (évite la saisie)")
    parser.add_argument("--no-input", action="store_true",
                        help="ne jamais demander le token de façon interactive")
    parser.add_argument("--no-push", action="store_true",
                        help="copier et commiter sans pousser")
    parser.add_argument("--prune-submissions", action="store_true",
                        help="supprimer les dossiers de soumission obsolètes (ancien slug) "
                             "avant publication : un seul slug est autorisé par PR")
    parser.add_argument("--include-submissions", action="store_true",
                        help="publier aussi submissions/**")
    parser.add_argument("--zip", action="store_true",
                        help="créer en plus une archive de secours dans --source")
    parser.add_argument("--force", action="store_true",
                        help="publier malgré les avertissements de conformité")
    parser.add_argument("--create-branch", action="store_true",
                        help="autoriser la création de la branche si elle n'existe pas")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    source = Path(args.source).resolve()
    branch = args.branch
    repo_url = args.repo_url

    log("=" * 74)
    log("Publication des artefacts vers GitHub")
    log("=" * 74)
    log(f"Source      : {source}")
    log(f"Dépôt       : {repo_url}")
    log(f"Branche     : {branch}")
    log(f"Artefacts   : {', '.join(ARTIFACT_DIRS + (('submissions',) if args.include_submissions else ()))}")
    log()

    if args.prune_submissions:
        prune_obsolete_submissions(source)

    files = collect_artifacts(source, args.include_submissions)
    if not files:
        die(f"aucun artefact trouvé dans {source} (attendu : models/, reports/)", EXIT_MISSING)

    log(f"{len(files)} fichier(s) à publier :")
    total = 0
    for rel in files:
        size = (source / rel).stat().st_size
        total += size
        log(f"  {size:>12,} o  {rel}")
    log(f"  {'-' * 12}")
    log(f"  {total:>12,} o  total")

    safety_checks(source, files, args.force)

    if args.zip:
        archive = source / "artifacts_backup.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            for rel in files:
                handle.write(source / rel, rel)
        log(f"\nArchive de secours : {archive} ({archive.stat().st_size:,} o)")

    token = read_token(args)

    repo = Path(args.repo).resolve() if args.repo else clone_dir_default()

    if not (repo / ".git").exists():
        if not token:
            die("aucun token fourni et pas de clone local : impossible de cloner.", EXIT_ERROR)
        log(f"\nClone de {repo_url} (branche {branch}) dans {repo} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", branch, authed_url(repo_url, token), str(repo)],
            capture_output=True, text=True)
        if clone.returncode != 0:
            die("clonage impossible (token invalide, branche inexistante ou réseau) :\n"
                f"{redact(clone.stderr or clone.stdout, token).strip()}", EXIT_ERROR)
        log("clone : OK")
    else:
        log(f"\nClone existant réutilisé : {repo}")

    if token:
        check = git(repo, "ls-remote", "--heads", authed_url(repo_url, token), branch)
        if check.returncode != 0:
            die("authentification refusée : vérifiez la portée `repo` du token,"
                " sa date d'expiration et le nom de la branche.", EXIT_ERROR)
        log("authentification : OK")

    # La branche cible doit exister : sans ce contrôle, une faute de frappe
    # créerait silencieusement une nouvelle branche distante.
    exists = git(None, "ls-remote", "--heads",
                 authed_url(repo_url, token) if token else repo_url, branch)
    if exists.returncode == 0 and not exists.stdout.strip():
        if args.create_branch:
            log(f"branche '{branch}' absente du dépôt : elle sera créée (--create-branch).")
        else:
            die(f"la branche '{branch}' n'existe pas sur {repo_url}.\n"
                "Vérifiez le nom (--branch), ou utilisez --create-branch pour la créer.",
                EXIT_ERROR)
    elif exists.returncode != 0 and not token:
        log("(impossible de vérifier la branche sans token : le push tranchera.)")

    # --- resynchronisation ---------------------------------------------------
    # Un clone Colab réutilisé (ou un clone créé dans une session précédente) peut être
    # en retard sur la branche distante : le commit local ne serait alors pas un
    # fast-forward et le push serait refusé. On se replace d'abord sur la tête distante ;
    # les artefacts étant recopiés juste après, rien n'est perdu.
    fetch = git(repo, "fetch", authed_url(repo_url, token) if token else repo_url, branch)
    if fetch.returncode == 0:
        ancestor = git(repo, "merge-base", "--is-ancestor", "FETCH_HEAD", "HEAD")
        if ancestor.returncode == 0:
            log("clone à jour avec la branche distante.")
        else:
            local = git(repo, "rev-parse", "--short", "HEAD").stdout.strip()
            remote = git(repo, "rev-parse", "--short", "FETCH_HEAD").stdout.strip()
            log(f"clone en retard ({local}) sur la branche distante ({remote}) : "
                "resynchronisation sur la tête distante (les artefacts sont recopiés ensuite).")
            git_or_die(repo, token, "checkout", "-B", branch, "FETCH_HEAD",
                       what=f"git checkout -B {branch} {remote}")
    else:
        log("fetch impossible (réseau ?) : on tente le push tel quel.")

    for rel in files:
        destination = repo / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, destination)
    log(f"{len(files)} fichier(s) copié(s) dans le clone.")

    git(repo, "config", "user.name", "Artifact Publisher")
    git(repo, "config", "user.email", "publisher@users.noreply.github.com")
    for root in {Path(rel).parts[0] for rel in files}:
        git_or_die(repo, token, "add", root, what=f"git add {root}")

    commit = git(repo, "commit", "-m", args.message)
    if commit.returncode == 0:
        log("commit : OK")
    elif "nothing to commit" in (commit.stdout + commit.stderr):
        log("commit : rien de nouveau (artefacts identiques)")
    else:
        die(f"commit impossible :\n{redact(commit.stderr or commit.stdout, token).strip()}",
            EXIT_ERROR)

    if args.no_push:
        log("\n--no-push : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    if not token:
        log("\nAucun token : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    push = git(repo, "push", authed_url(repo_url, token), f"HEAD:{branch}")
    if push.returncode != 0:
        die(f"push refusé :\n{redact(push.stderr or push.stdout, token).strip()}", EXIT_ERROR)

    log("push : OK")
    log()
    log(f"Publié sur {repo_url} (branche {branch}).")
    if "github.com" in repo_url:
        slug = repo_url.rstrip("/").removesuffix(".git")
        log(f"Vérifiez : {slug}/tree/{branch}")
    return EXIT_OK


if __name__ == "__main__":
    raise SystemExit(main())


### Exécution (token masqué)

In [ ]:
# =============================================================================
# Exécution du script de publication (champ masqué Colab actif)
# =============================================================================
import runpy
import sys
from pathlib import Path

# Cellule autonome : elle fonctionne même si elle est la SEULE exécutée après un
# redémarrage de la VM (OUTPUT_ROOT, défini plus haut, n'existe alors pas encore).
SOURCE = Path(globals().get("OUTPUT_ROOT") or Path.cwd())
if not (SOURCE / "reports").is_dir():
    for candidate in (Path.cwd(), Path("/content")):
        if (candidate / "reports").is_dir():
            SOURCE = candidate
            break

SCRIPT_PATH = None
for candidate in (Path.cwd() / "push_artifacts_to_github.py",
                  SOURCE / "push_artifacts_to_github.py",
                  Path("/content/push_artifacts_to_github.py")):
    if candidate.is_file():
        SCRIPT_PATH = candidate.resolve()
        break
assert SCRIPT_PATH is not None, "la cellule %%writefile ci-dessus doit être exécutée d'abord"

# --prune-submissions : supprime un éventuel dossier de soumission obsolète (ancien slug),
# car le checker officiel exige exactement un répertoire submissions/<slug>/.
sys.argv = [
    "push_artifacts_to_github.py",
    "--source", str(SOURCE),
    "--branch", "arena/01a0889d-tokenizer",
    "--include-submissions",
    "--prune-submissions",
]

print("Exécution :", SCRIPT_PATH)
print("Arguments :", " ".join(sys.argv[1:]))
print("Un champ masqué « Colle ton token GitHub puis valide » va s'afficher.")
print()
try:
    runpy.run_path(str(SCRIPT_PATH), run_name="__main__")
except SystemExit as exc:
    print()
    print("Code de sortie du script :", exc.code,
          "| 0 = OK, 1 = erreur, 2 = artefacts manquants, 3 = publication refusée")
